# Notebook 2 — Prepare the Model for the App

In Notebook 1, we tested ways to detect suspicious network connections.
Random Forest caught most attacks, but it also produced many false alarms.

In this notebook, we will save the selected model and the information the
Streamlit app needs to use it. The app will show a prediction as a
**possible warning**, not proof that a connection is an attack.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("C:/Projects/21_Cyber_Threat_Detection/data/UNSW_NB15_training-set.csv")
df = pd.read_csv(data_path)

In [3]:
# Seperate clues from answers
# ( X contain clues; y contains the correct answer)
X = df.drop(columns=["id", "attack_cat", "label"])
y = df["label"]

print("Clues:", X.shape)
print("Answers:", y.shape)

Clues: (175341, 42)
Answers: (175341,)


In [4]:
# Convert text clues to numbers
# Create a numeric column for each text category
X_encoded = pd.get_dummies(
    X,
    columns=["proto", "service", "state"],
    dtype="int8"
)

print("Before:", X.shape)
print("After:", X_encoded.shape)

Before: (175341, 42)
After: (175341, 194)


In [5]:
# Train the app's model
# Train on the full training dataset for use in the app
from sklearn.ensemble import RandomForestClassifier

final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_encoded, y)

print("Model trained on", len(X_encoded), "connections.")

Model trained on 175341 connections.


In [6]:
# Save the model
# Save everything the app needs for prediction
import joblib

model_path = Path("C:/Projects/21_Cyber_Threat_Detection/models/network_detector.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "model": final_model,
        "input_columns": X_encoded.columns.tolist(),
        "text_columns": ["proto", "service", "state"],
    },
    model_path
)

print("Saved to:", model_path)

Saved to: C:\Projects\21_Cyber_Threat_Detection\models\network_detector.joblib


In [8]:
# Check the saved  model
# Load the saved model and predict one training example
saved = joblib.load(model_path)

one_connection = X.iloc[[0]]
one_encoded = pd.get_dummies(
    one_connection,
    columns=saved["text_columns"],
    dtype="int8"
)
one_encoded = one_encoded.reindex(
    columns=saved["input_columns"],
    fill_value=0
)

prediction = saved["model"].predict(one_encoded)[0]

print("Model prediction:", prediction)
print("Known answer:", y.iloc[0])


Model prediction: 0
Known answer: 0


In [9]:
# Find the test file
# Show test CSV files in the data folder
data_dir = data_path.parent
print([file.name for file in data_dir.glob("*test*.csv")])

['UNSW_NB15_testing-set.csv']


In [10]:
# Evaluate the saved model
# Load and prepare the official test data
from sklearn.metrics import precision_score, recall_score

test_df = pd.read_csv(data_dir / "UNSW_NB15_testing-set.csv")
test_y = test_df["label"]

test_clues = test_df.drop(columns=["id", "attack_cat", "label"])
test_encoded = pd.get_dummies(
    test_clues,
    columns=saved["text_columns"],
    dtype="int8"
)
test_encoded = test_encoded.reindex(
    columns=saved["input_columns"],
    fill_value=0
)

# Predict and count correct answers and mistakes
test_predictions = saved["model"].predict(test_encoded)

display(pd.crosstab(
    test_y,
    test_predictions,
    rownames=["Actual (0=normal, 1=attack)"],
    colnames=["Predicted (0=normal, 1=attack)"]
))

print("Precision:", round(precision_score(test_y, test_predictions), 3))
print("Recall:", round(recall_score(test_y, test_predictions), 3))


"Predicted (0=normal, 1=attack)",0,1
"Actual (0=normal, 1=attack)",,
0,25436,11564
1,358,44974


Precision: 0.795
Recall: 0.992


## Final model and limitation

The saved Random Forest was trained on the full UNSW-NB15 training set.
It accepts 194 encoded connection features.

On the separate official test set, it achieved:

- Precision: 79.5%
- Recall: 99.2%
- False alarms: 11,564 normal connections flagged as attacks
- Missed attacks: 358

The model's prediction is a possible warning, not proof of an attack.
It has too many false alarms for unattended real-world security use.

In [11]:
# See how a 0.7 warning threshold performs on separate test data
from sklearn.metrics import confusion_matrix

attack_scores = saved["model"].predict_proba(test_encoded)[:, 1]
warnings_07 = (attack_scores >= 0.7).astype(int)

tn, fp, fn, tp = confusion_matrix(test_y, warnings_07).ravel()

print("False alarms:", fp)
print("Missed attacks:", fn)
print("Attacks caught:", tp)
print("Normal connections correctly left alone:", tn)

False alarms: 5205
Missed attacks: 2182
Attacks caught: 43150
Normal connections correctly left alone: 31795


In [12]:
# Train XGBoost on the full training dataset
from xgboost import XGBClassifier

final_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

final_xgb.fit(X_encoded, y)

print("XGBoost trained on", len(X_encoded), "connections.")

XGBoost trained on 175341 connections.


In [13]:
# Count connections flagged only by XGBoost
from sklearn.metrics import confusion_matrix

rf_flags = saved["model"].predict(test_encoded)
xgb_flags = final_xgb.predict(test_encoded)

extra_review = (rf_flags == 0) & (xgb_flags == 1)
combined_flags = ((rf_flags == 1) | (xgb_flags == 1)).astype(int)

tn, fp, fn, tp = confusion_matrix(test_y, combined_flags).ravel()

print("Additional connections for review:", extra_review.sum())
print("  Actual attacks recovered:", ((test_y == 1) & extra_review).sum())
print("  Extra false alarms:", ((test_y == 0) & extra_review).sum())
print("Total false alarms with both models:", fp)
print("Total missed attacks with both models:", fn)

Additional connections for review: 754
  Actual attacks recovered: 112
  Extra false alarms: 642
Total false alarms with both models: 12206
Total missed attacks with both models: 246


## XGBoost comparison

We tested XGBoost as a second model to review connections that Random
Forest predicted as normal.

On the official test set, it identified 112 additional attacks but also
flagged 642 additional normal connections. The combined system missed
246 attacks and produced 12,206 false alarms.

Decision: Keep Random Forest as the app's default detector. The extra
XGBoost review queue creates too many additional false alarms for the
number of attacks it recovers. This comparison is exploratory because
the official test set was used during model decisions.

In [14]:
# Check whether the model file can fit in the GitHub repository
size_mb = model_path.stat().st_size / (1024 * 1024)
print(f"Model size: {size_mb:.1f} MB")

Model size: 14.0 MB


In [15]:
# Check the packages used to create and load the saved model
import importlib.metadata as metadata

for package in ["streamlit", "pandas", "scikit-learn", "joblib"]:
    print(package, metadata.version(package))

streamlit 1.45.1
pandas 2.2.3
scikit-learn 1.6.1
joblib 1.4.2
